<a href="https://colab.research.google.com/github/ManasDasri/NNDL/blob/main/notebooks/4_problem2_longformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4 · Problem 2 — does longer context help?

BERT at 512 tokens against Longformer at 4,096. Both train on chunks, because 70%
of contracts exceed 4,096 as well. What differs is how much context one window
holds: about 300 words against about 2,600.

**Needs a GPU with real memory.** Longformer at 4,096 tokens wants a small batch
and gradient checkpointing; budget a few hours on a T4.


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [1]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


/content/NNDL
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for legal-risk-clause-classifier (pyproject.toml) ... done
GPU: Tesla T4


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [2]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


510 documents -> data/processed/documents.jsonl
splits {'train': 358, 'val': 76, 'test': 76} -> data/processed/splits.json
  Cap on Liability                   662 spans across  275 documents
  Non-Compete                        251 spans across  119 documents
  License Grant                      772 spans across  255 documents
  Audit Rights                       639 spans across  214 documents
  Termination for Convenience        245 spans across  183 documents
  Insurance                          554 spans across  166 documents


### Train Longformer

`[CLS]` is given global attention so the classification head reads a token that
attended to the whole window. Without it the head sees only a local neighbourhood
and the long context is wasted on the one position that matters.


In [3]:
!legal-risk-train-transformer --model longformer --epochs 3 \
    --batch_size 1 --grad_accumulation 16 --lr 2e-5 \
    --gradient_checkpointing --output_dir outputs/longformer


allenai/longformer-base-4096  window=2600w overlap=400w max_length=4096t
{'train': 1380, 'val': 295, 'test': 326}
config.json: 100% 694/694 [00:00<00:00, 3.92MB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 38.1MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 76.0MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 121MB/s]

pytorch_model.bin: downloading bytes:  25% 149M/597M [00:01<00:02, 201MB/s, 10.8MB/s  ]
pytorch_model.bin: downloading bytes:  32% 191M/597M [00:01<00:02, 169MB/s, 17.0MB/s  ]
pytorch_model.bin: downloading bytes:  37% 219M/597M [00:01<00:02, 160MB/s, 19.1MB/s  ]
pytorch_model.bin: downloading bytes:  46% 274M/597M [00:01<00:01, 198MB/s, 22.8MB/s  ]
pytorch_model.bin: downloading bytes:  58% 347M/597M [00:02<00:01, 224MB/s, 28.4MB/s  ]
pytorch_model.bin: reconstructing file:  71% 422M/597M [00:02<00:00, 244MB/s, 32.5MB/s  ]
pytorch_model.bin: reconstructing file:  86% 513M/597M [00:02<00:00, 292MB/s, 41.1MB/s  ]
pytorch_model.bin: downloading bytes: 100% 363M/363M [00:

### Compare against the 512-token model


In [4]:
# Compare whatever has finished; legal-risk-compare skips missing runs.
!legal-risk-evaluate --run_dir outputs/longformer --split test
!legal-risk-compare outputs/cnn outputs/bert outputs/legal_bert outputs/longformer


Loading weights: 100% 269/269 [00:00<00:00, 21216.42it/s]
[transformers] LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.dense.weight           | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
classifier.dense.weight        | MISSING    | 
classifier.out_proj.bias       | MISSING    | 
classifier.dense.bias          | MISSING    | 
classifier.out_proj.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fr

In [5]:
print(open('docs/results.md').read())


# Model comparison

Generated by `legal-risk-compare` from the runs themselves, on the test split.
Every model saw identical chunks, identical document-level splits, the same
weighted loss and the same per-label threshold tuning, so the differences below
are differences between models.

## Chunk-level results

| Model | Context | Chunk words | macro F1 | macro AP | Best epoch |
| --- | ---: | ---: | ---: | ---: | ---: |
| allenai/longformer-base-4096 | 4096 | 2600 | 0.551 | 0.578 | 3 |

## Per-class F1

| Model | Cap on Liability | Non-Compete | License Grant | Audit Rights | Termination for Convenience | Insurance |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| longformer | 0.710 | 0.346 | 0.754 | 0.436 | 0.203 | 0.857 |

## Document-level results

Chunk predictions pooled to a contract-level decision. This is the number that
matters for the tool as it would be used: does this contract contain the clause?

| Model | Pooling | macro F1 | macro AP |
| --- | --- | ---: | ---: |
| l

### Cost, which belongs in the comparison

Longformer is larger and much slower. If it wins by a small margin, the honest
reporting includes what that margin cost.


In [6]:
import json, pathlib

for name in ('bert', 'legal_bert', 'longformer'):
    path = pathlib.Path('outputs') / name / 'metrics.json'
    if not path.exists():
        print(f'{name:12s} not trained yet')
        continue
    m = json.load(open(path))
    print(f"{name:12s} {m['parameters']:>12,} params  "
          f"window {m['chunking']['window']:>5} words  "
          f"train windows {m['chunks']['train']:>6,}  "
          f"macro F1 {m['test_at_tuned_thresholds']['macro_f1']:.3f}")


bert         not trained yet
legal_bert   not trained yet
longformer    148,664,070 params  window  2600 words  train windows  1,380  macro F1 0.551


### A null result is a result

Most of these clauses are locally signalled — *"shall maintain insurance"* is
recognisable from its own sentence. If Longformer does not win, that is a finding
about the task: clause detection in contracts is a local problem, and the expensive
long-context machinery is not what it needs. Say that plainly rather than treating
it as a failed experiment.
